In [1]:
import nbformat

nb_path = "Main_Pairwise.ipynb"

nb = nbformat.read(nb_path, as_version=4)

if "tags" not in nb.cells[0].metadata:
    nb.cells[0].metadata["tags"] = []

if "parameters" not in nb.cells[0].metadata["tags"]:
    nb.cells[0].metadata["tags"].append("parameters")

nbformat.write(nb, nb_path)

print("Added 'parameters' tag to first cell.")

Added 'parameters' tag to first cell.


In [2]:
# =============================================================================
# BUILD ALL PAIRWISE M1 DATASETS WITH RANDOM LABEL RESOLUTION
# =============================================================================
# Purpose:
#   - Create all pairwise concatenated datasets from:
#       Pre_M1, Post1_M1, Post2_M1, T2_M1, ADC_M1
#   - Align samples by INFO_NameOfRoi.
#   - Prefix feature columns by modality name.
#   - If one ROI has different labels across datasets, randomly choose one label.
#   - Use fixed random seed for reproducibility.
# =============================================================================

import os
import re
import itertools
import random
import numpy as np
import pandas as pd

# -----------------------------
# Configuration
# -----------------------------

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DATASETS = {
    "Pre":   "../Dataset/Breast-Data/Mask1/Pre_M1.csv",
    "Post1": "../Dataset/Breast-Data/Mask1/Post1_M1.csv",
    "Post2": "../Dataset/Breast-Data/Mask1/Post2_M1.csv",
    "T2":    "../Dataset/Breast-Data/Mask1/T2_M1.csv",
    "ADC":   "../Dataset/Breast-Data/Mask1/ADC_M1.csv",
}

OUT_DIR = "./output/pairwise_datasets_random_label_resolution"
os.makedirs(OUT_DIR, exist_ok=True)

ID_COL = "INFO_NameOfRoi"


# -----------------------------
# Helper functions
# -----------------------------

def standardize_label_column(df):
    """
    Standardize Label/Lable column name to Label.
    """
    df = df.copy()

    if "Label" not in df.columns and "Lable" in df.columns:
        df = df.rename(columns={"Lable": "Label"})

    if "Label" not in df.columns:
        raise ValueError("No Label or Lable column found.")

    return df


def extract_patient_id_from_roi(roi):
    """
    Extract patient ID from ROI name.
    Example:
        M1_S2_P17_R -> P17
        M1_S1_P12_R_#1 -> P12
    """
    roi = str(roi)
    match = re.search(r"P(\d+)", roi)
    if match:
        return f"P{match.group(1)}"
    return np.nan


def load_single_dataset(dataset_name, path):
    """
    Load one radiomics CSV file and prepare it:
      - Standardize label column.
      - Extract PatientID.
      - Keep numeric feature columns.
      - Prefix feature columns with dataset name.
    """
    df = pd.read_csv(path)
    df = standardize_label_column(df)

    if ID_COL not in df.columns:
        raise ValueError(f"{ID_COL} not found in {dataset_name}")

    df["PatientID"] = df[ID_COL].apply(extract_patient_id_from_roi)

    non_feature_cols = {
        "INFO_PatientName",
        "INFO_NameOfRoi",
        "Unnamed: 2",
        "FEATURE RESULTS",
        "Label",
        "Lable",
        "PatientID",
    }

    candidate_feature_cols = [
        c for c in df.columns
        if c not in non_feature_cols
    ]

    # Convert feature columns to numeric
    feature_df = df[candidate_feature_cols].apply(pd.to_numeric, errors="coerce")

    # Drop columns that are fully non-numeric / empty
    feature_df = feature_df.dropna(axis=1, how="all")

    # Prefix feature names
    feature_df = feature_df.add_prefix(f"{dataset_name}__")

    prepared_df = pd.concat(
        [
            df[[ID_COL, "PatientID", "Label"]].reset_index(drop=True),
            feature_df.reset_index(drop=True)
        ],
        axis=1
    )

    # Check duplicated ROI IDs
    if prepared_df[ID_COL].duplicated().any():
        duplicated_rois = prepared_df.loc[
            prepared_df[ID_COL].duplicated(keep=False),
            ID_COL
        ].tolist()

        raise ValueError(
            f"Duplicate ROI IDs found in {dataset_name}: {duplicated_rois[:10]}"
        )

    return prepared_df


# -----------------------------
# Load all datasets
# -----------------------------

prepared = {}

for name, path in DATASETS.items():
    prepared[name] = load_single_dataset(name, path)
    print(
        f"{name}: "
        f"samples={prepared[name].shape[0]}, "
        f"features={prepared[name].shape[1] - 3}, "
        f"patients={prepared[name]['PatientID'].nunique()}"
    )


# -----------------------------
# Create global random label map
# -----------------------------
# Important:
#   If an ROI has mismatch across modalities, we choose ONE random label globally.
#   This prevents the same ROI from getting different labels in different pairwise datasets.

roi_label_records = {}

for name, df in prepared.items():
    for _, row in df.iterrows():
        roi = row[ID_COL]
        label = row["Label"]

        if roi not in roi_label_records:
            roi_label_records[roi] = {}

        roi_label_records[roi][name] = label


resolved_label_map = {}
mismatch_report_rows = []

for roi, labels_by_dataset in roi_label_records.items():

    labels_present = [
        labels_by_dataset[d]
        for d in labels_by_dataset
        if pd.notna(labels_by_dataset[d])
    ]

    unique_labels = sorted(set(labels_present))

    if len(unique_labels) == 1:
        resolved_label = unique_labels[0]
        resolution_method = "consistent"
    else:
        resolved_label = random.choice(unique_labels)
        resolution_method = "random_selected"

        report_row = {
            "INFO_NameOfRoi": roi,
            "PatientID": extract_patient_id_from_roi(roi),
            "Unique_Original_Labels": ",".join(map(str, unique_labels)),
            "Random_Selected_Label": resolved_label,
            "Random_Seed": RANDOM_SEED,
            "Resolution_Method": "global_per_roi_random_choice"
        }

        for dataset_name in DATASETS.keys():
            report_row[f"Original_Label_{dataset_name}"] = labels_by_dataset.get(dataset_name, np.nan)

        mismatch_report_rows.append(report_row)

    resolved_label_map[roi] = resolved_label


df_mismatch_report = pd.DataFrame(mismatch_report_rows)

mismatch_report_path = os.path.join(
    OUT_DIR,
    "global_random_label_resolution_report.csv"
)

df_mismatch_report.to_csv(mismatch_report_path, index=False)

print("\nUnique ROI-level mismatches randomly resolved:", len(df_mismatch_report))

if len(df_mismatch_report) > 0:
    display(df_mismatch_report)


# -----------------------------
# Build all pairwise datasets
# -----------------------------

pairwise_summary = []

for name_a, name_b in itertools.combinations(DATASETS.keys(), 2):

    df_a = prepared[name_a].copy()
    df_b = prepared[name_b].copy()

    # Merge by ROI
    merged = df_a.merge(
        df_b,
        on=ID_COL,
        how="inner",
        suffixes=(f"_{name_a}", f"_{name_b}")
    )

    # Count original mismatches in this pair
    pair_mismatch_mask = (
        merged[f"Label_{name_a}"] != merged[f"Label_{name_b}"]
    )

    n_pair_mismatches = int(pair_mismatch_mask.sum())

    # Use global random-resolved label
    merged["Label"] = merged[ID_COL].map(resolved_label_map)

    # Use PatientID from first dataset
    merged["PatientID"] = merged[f"PatientID_{name_a}"]

    # Remove duplicate metadata columns
    cols_to_drop = [
        f"PatientID_{name_a}",
        f"PatientID_{name_b}",
        f"Label_{name_a}",
        f"Label_{name_b}",
    ]

    merged = merged.drop(columns=cols_to_drop)

    # Reorder columns
    feature_cols = [
        c for c in merged.columns
        if c not in [ID_COL, "PatientID", "Label"]
    ]

    final_df = merged[[ID_COL, "PatientID", "Label"] + feature_cols].copy()

    # Save output
    pair_name = f"{name_a}_{name_b}_M1_pairwise_random_label.csv"
    pair_path = os.path.join(OUT_DIR, pair_name)

    final_df.to_csv(pair_path, index=False)

    class_counts = final_df["Label"].value_counts().to_dict()

    pairwise_summary.append({
        "Pair": f"{name_a}+{name_b}",
        "Output_File": pair_name,
        "n_common_roi": final_df.shape[0],
        "n_label_mismatches_random_resolved_in_pair": n_pair_mismatches,
        "n_samples_final": final_df.shape[0],
        "n_patients_final": final_df["PatientID"].nunique(),
        "n_features_final": len(feature_cols),
        "class_0": class_counts.get(0, class_counts.get("0", 0)),
        "class_1": class_counts.get(1, class_counts.get("1", 0)),
        "Path": pair_path
    })

    print(
        f"{name_a}+{name_b}: "
        f"samples={final_df.shape[0]}, "
        f"patients={final_df['PatientID'].nunique()}, "
        f"features={len(feature_cols)}, "
        f"random_resolved_mismatches={n_pair_mismatches}"
    )


# -----------------------------
# Save summary
# -----------------------------

df_pairwise_summary = pd.DataFrame(pairwise_summary)

summary_path = os.path.join(
    OUT_DIR,
    "pairwise_random_label_summary.csv"
)

df_pairwise_summary.to_csv(summary_path, index=False)

print("\nSaved pairwise datasets to:")
print(OUT_DIR)

print("\nSaved summary to:")
print(summary_path)

print("\nSaved random label resolution report to:")
print(mismatch_report_path)

display(df_pairwise_summary)

Pre: samples=117, features=145, patients=48
Post1: samples=116, features=145, patients=48
Post2: samples=117, features=145, patients=48
T2: samples=117, features=145, patients=48
ADC: samples=114, features=144, patients=48

Unique ROI-level mismatches randomly resolved: 5


,INFO_NameOfRoi,PatientID,Unique_Original_Labels,Random_Selected_Label,Random_Seed,Resolution_Method,Original_Label_Pre,Original_Label_Post1,Original_Label_Post2,Original_Label_T2,Original_Label_ADC
0,M1_S4_P12_L_#1,P12,"0,1",0,42,global_per_roi_random_choice,1,1,1,0,1
1,M1_S3_P12_L_#1,P12,"0,1",0,42,global_per_roi_random_choice,1,1,1,0,1
2,M1_S2_P12_R_#1,P12,"0,1",1,42,global_per_roi_random_choice,0,0,0,1,0
3,M1_S1_P12_R_#1,P12,"0,1",0,42,global_per_roi_random_choice,0,0,0,1,0
4,M1_S2_P17_R,P17,"0,1",0,42,global_per_roi_random_choice,0,0,1,0,0


Pre+Post1: samples=116, patients=48, features=290, random_resolved_mismatches=0
Pre+Post2: samples=117, patients=48, features=290, random_resolved_mismatches=1
Pre+T2: samples=117, patients=48, features=290, random_resolved_mismatches=4
Pre+ADC: samples=114, patients=48, features=289, random_resolved_mismatches=0
Post1+Post2: samples=116, patients=48, features=290, random_resolved_mismatches=1
Post1+T2: samples=116, patients=48, features=290, random_resolved_mismatches=4
Post1+ADC: samples=113, patients=48, features=289, random_resolved_mismatches=0
Post2+T2: samples=117, patients=48, features=290, random_resolved_mismatches=5
Post2+ADC: samples=114, patients=48, features=289, random_resolved_mismatches=1
T2+ADC: samples=114, patients=48, features=289, random_resolved_mismatches=4

Saved pairwise datasets to:
./output/pairwise_datasets_random_label_resolution

Saved summary to:
./output/pairwise_datasets_random_label_resolution/pairwise_random_label_summary.csv

Saved random label reso

,Pair,Output_File,n_common_roi,n_label_mismatches_random_resolved_in_pair,n_samples_final,n_patients_final,n_features_final,class_0,class_1,Path
0,Pre+Post1,Pre_Post1_M1_pairwise_random_label.csv,116,0,116,48,290,53,63,./output/pairwise_datasets_random_label_resolu...
1,Pre+Post2,Pre_Post2_M1_pairwise_random_label.csv,117,1,117,48,290,53,64,./output/pairwise_datasets_random_label_resolu...
2,Pre+T2,Pre_T2_M1_pairwise_random_label.csv,117,4,117,48,290,53,64,./output/pairwise_datasets_random_label_resolu...
3,Pre+ADC,Pre_ADC_M1_pairwise_random_label.csv,114,0,114,48,289,51,63,./output/pairwise_datasets_random_label_resolu...
4,Post1+Post2,Post1_Post2_M1_pairwise_random_label.csv,116,1,116,48,290,53,63,./output/pairwise_datasets_random_label_resolu...
5,Post1+T2,Post1_T2_M1_pairwise_random_label.csv,116,4,116,48,290,53,63,./output/pairwise_datasets_random_label_resolu...
6,Post1+ADC,Post1_ADC_M1_pairwise_random_label.csv,113,0,113,48,289,51,62,./output/pairwise_datasets_random_label_resolu...
7,Post2+T2,Post2_T2_M1_pairwise_random_label.csv,117,5,117,48,290,53,64,./output/pairwise_datasets_random_label_resolu...
8,Post2+ADC,Post2_ADC_M1_pairwise_random_label.csv,114,1,114,48,289,51,63,./output/pairwise_datasets_random_label_resolu...
9,T2+ADC,T2_ADC_M1_pairwise_random_label.csv,114,4,114,48,289,51,63,./output/pairwise_datasets_random_label_resolu...


In [3]:
import os
import glob
from pathlib import Path

try:
    import papermill as pm
except ImportError:
    !pip install -q papermill
    import papermill as pm

TEMPLATE_NOTEBOOK = "Main_Pairwise.ipynb"

PAIRWISE_DIR = "output/pairwise_datasets_random_label_resolution"

BATCH_ROOT = "./nested_cv_outputs_pairwise_batch"
EXECUTED_NOTEBOOK_DIR = "./executed_pairwise_notebooks"

os.makedirs(BATCH_ROOT, exist_ok=True)
os.makedirs(EXECUTED_NOTEBOOK_DIR, exist_ok=True)

pairwise_files = sorted(
    glob.glob(os.path.join(PAIRWISE_DIR, "*_M1_pairwise_random_label.csv"))
)

print("Found pairwise datasets:", len(pairwise_files))
for f in pairwise_files:
    print(" -", f)

completed_runs = []
failed_runs = []

for input_file in pairwise_files:
    
    dataset_tag = Path(input_file).stem.replace("_M1_pairwise_random_label", "")
    
    output_notebook = os.path.join(
        EXECUTED_NOTEBOOK_DIR,
        f"{dataset_tag}_executed.ipynb"
    )
    
    print("\n" + "=" * 100)
    print("Running:", dataset_tag)
    print("Input:", input_file)
    print("Executed notebook:", output_notebook)
    print("=" * 100)
    
    try:
        pm.execute_notebook(
            input_path=TEMPLATE_NOTEBOOK,
            output_path=output_notebook,
            parameters={
                "INPUT_FILE": input_file,
                "DATASET_TAG": dataset_tag,
                "BATCH_ROOT": BATCH_ROOT,
                "OUTPUT_DIR": f"./output/cleaned_pairwise/{dataset_tag}",
            },
            kernel_name="python3",
            progress_bar=True,
            log_output=True,
        )
        
        completed_runs.append(dataset_tag)
        print("DONE:", dataset_tag)
        
    except Exception as e:
        failed_runs.append({
            "dataset": dataset_tag,
            "input_file": input_file,
            "error": str(e)
        })
        print("FAILED:", dataset_tag)
        print(e)

print("\nBATCH FINISHED")
print("Completed:", completed_runs)
print("Failed:", failed_runs)

Found pairwise datasets: 10
 - output/pairwise_datasets_random_label_resolution/Post1_ADC_M1_pairwise_random_label.csv
 - output/pairwise_datasets_random_label_resolution/Post1_Post2_M1_pairwise_random_label.csv
 - output/pairwise_datasets_random_label_resolution/Post1_T2_M1_pairwise_random_label.csv
 - output/pairwise_datasets_random_label_resolution/Post2_ADC_M1_pairwise_random_label.csv
 - output/pairwise_datasets_random_label_resolution/Post2_T2_M1_pairwise_random_label.csv
 - output/pairwise_datasets_random_label_resolution/Pre_ADC_M1_pairwise_random_label.csv
 - output/pairwise_datasets_random_label_resolution/Pre_Post1_M1_pairwise_random_label.csv
 - output/pairwise_datasets_random_label_resolution/Pre_Post2_M1_pairwise_random_label.csv
 - output/pairwise_datasets_random_label_resolution/Pre_T2_M1_pairwise_random_label.csv
 - output/pairwise_datasets_random_label_resolution/T2_ADC_M1_pairwise_random_label.csv

Running: Post1_ADC
Input: output/pairwise_datasets_random_label_resol

Executing:   0%|          | 0/59 [00:00<?, ?cell/s]

DONE: Post1_ADC

Running: Post1_Post2
Input: output/pairwise_datasets_random_label_resolution/Post1_Post2_M1_pairwise_random_label.csv
Executed notebook: ./executed_pairwise_notebooks/Post1_Post2_executed.ipynb


Executing:   0%|          | 0/59 [00:00<?, ?cell/s]

PermutationExplainer explainer:  99%|█████████▉| 115/116 [00:00<?, ?it/s]
PermutationExplainer explainer: 117it [00:10,  5.07s/it]                 




...

Executing:   0%|          | 0/59 [00:00<?, ?cell/s]

DONE: Pre_T2

Running: T2_ADC
Input: output/pairwise_datasets_random_label_resolution/T2_ADC_M1_pairwise_random_label.csv
Executed notebook: ./executed_pairwise_notebooks/T2_ADC_executed.ipynb


Executing:   0%|          | 0/59 [00:00<?, ?cell/s]

PermutationExplainer explainer: 100%|██████████| 114/114 [00:12<00:00, 11.13it/s]
PermutationExplainer explainer: 115it [00:12,  2.33it/s]                         




DONE: T2_ADC

BATCH FINISHED
Completed: ['Post1_ADC', 'Post1_Post2', 'Post1_T2', 'Post2_ADC', 'Post2_T2', 'Pre_ADC', 'Pre_Post1', 'Pre_Post2', 'Pre_T2', 'T2_ADC']
Failed: []
